# 🎬 Story Director 27B — Resume Verification Demo

**Purpose:** Verify that `RESUME=True` correctly skips epoch 1 and starts epoch 2 from your saved checkpoint — before committing to the full ~2hr run.

**What this does:**
1. Takes 60 examples from your real `training_data_v9.jsonl`
2. Runs epoch 1 → saves checkpoint
3. Resumes → confirms first **new** log entry shows `epoch > 1.0` ✅
4. Verifies checkpoint files are complete (no silent data loss)

**Runtime:** ~3–5 min on H100

---
**✅ PASS:** First new log entry shows `epoch: 1.XX` → resume works, epoch 1 skipped

**❌ FAIL:** First new log entry shows `epoch: 0.XX` → checkpoint path wrong or files missing

> **Note:** `trainer.state.log_history` after resume includes epoch 1 history loaded from the checkpoint file. The verdict filters by `step > checkpoint_step` to only look at genuinely new entries.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ═══════════════════════════════════════════════════════════════
!pip install -q unsloth trl transformers datasets accelerate bitsandbytes
print('✅ Dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Mount Google Drive
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os

# Quick sanity check — confirm dataset is visible before continuing
DATASET_PATH = '/content/drive/MyDrive/ytllm_v2/training_data_v9.jsonl'

if os.path.exists(DATASET_PATH):
    size_mb = os.path.getsize(DATASET_PATH) / 1e6
    print(f'✅ Dataset found: {DATASET_PATH}  ({size_mb:.1f} MB)')
else:
    print(f'❌ File not found: {DATASET_PATH}')
    print()
    base = '/content/drive/MyDrive/ytllm_v2'
    if os.path.exists(base):
        print(f'Folder exists. Contents of {base}:')
        for f in sorted(os.listdir(base)):
            print(f'  {f}')
        print()
        print('Update DATASET_PATH in Cell 3 to match the correct filename.')
    else:
        print(f'Folder does not exist: {base}')
        print('Check your Drive and update DATASET_PATH in Cell 3.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Config & GPU check
# ═══════════════════════════════════════════════════════════════
import os, glob, json, torch
from datasets import Dataset

# ── Paths (edit these to match your Drive) ───────────────────
DATASET_PATH        = "/content/drive/MyDrive/ytllm_v2/training_data_v9.jsonl"
DEMO_CHECKPOINT_DIR = "/content/drive/MyDrive/ytllm_v2/demo_checkpoints_resume_test"

# ── Demo slice ───────────────────────────────────────────────
DEMO_SLICE      = 60
MAX_SEQ_LENGTH  = 2048
ENABLE_THINKING = False

os.makedirs(DEMO_CHECKPOINT_DIR, exist_ok=True)

# ── GPU check ────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found — switch to H100 runtime')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU : {gpu_name}')
print(f'   VRAM: {vram_gb:.1f} GB')
print(f'   Demo checkpoint dir: {DEMO_CHECKPOINT_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Load dataset slice (integrity check, real data untouched)
# ═══════════════════════════════════════════════════════════════

# Count total lines in full dataset first
total_lines = 0
bad_lines   = 0
with open(DATASET_PATH, 'r') as f:
    for line in f:
        total_lines += 1
        try:
            obj = json.loads(line)
            if 'messages' not in obj:
                bad_lines += 1
        except json.JSONDecodeError:
            bad_lines += 1

print(f'📊 Full dataset  : {total_lines} lines total')
print(f'   Invalid lines : {bad_lines}')
print(f'   Valid lines   : {total_lines - bad_lines}')
print()

# Load demo slice (first DEMO_SLICE valid examples only)
raw = []
with open(DATASET_PATH, 'r') as f:
    for line in f:
        if len(raw) >= DEMO_SLICE:
            break
        try:
            obj = json.loads(line)
            if 'messages' in obj and len(obj['messages']) >= 2:
                raw.append(obj)
        except json.JSONDecodeError:
            pass

effective_batch = 2 * 4  # per_device_batch=2 x grad_accum=4
steps_per_epoch = len(raw) // effective_batch

print(f'✅ Demo slice     : {len(raw)} examples (from first {DEMO_SLICE} valid)')
print(f'   Eff batch      : {effective_batch}')
print(f'   Steps / epoch  : ~{steps_per_epoch}')
print(f'   Epoch 1 ckpt   : step ~{steps_per_epoch}')
print(f'   After resume   : first log must show epoch > 1.0  ← the test')
print()
print(f'   ⚠️  Full dataset ({total_lines} lines) is never modified — demo uses a read-only slice')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Load model (skips reload if already in memory)
# ═══════════════════════════════════════════════════════════════
try:
    model
    tokenizer
    print('✅ Model already loaded — skipping reload')
except NameError:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = 'unsloth/Qwen3.6-27B',
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit   = True,
        dtype          = None,
    )
    tokenizer = get_chat_template(tokenizer, chat_template='qwen-3')

    # Smaller LoRA for demo speed (real run uses r=64)
    model = FastLanguageModel.get_peft_model(
        model,
        r              = 16,
        lora_alpha     = 16,
        target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_dropout   = 0,
        bias           = 'none',
        use_gradient_checkpointing = 'unsloth',
    )
    print('✅ Model loaded with demo LoRA config (r=16)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Format dataset with Story Director system prompt
# ═══════════════════════════════════════════════════════════════

SYSTEM_PROMPT = (
    'You are the Story Director for a 3D pixel art YouTube Shorts channel. '
    'You write viral short-form video scripts using the Hook → Foreshadow → '
    'Obstacle → Amplifier → Twist → Payoff framework. '
    'Rules you never break:\n'
    '- Hook lands in 1.5-3 seconds — it must create instant curiosity or stakes\n'
    '- Foreshadow is always 2 lines, under 3 seconds\n'
    '- Use but/therefore storytelling — never and/then\n'
    '- 90%+ retention is the minimum; 100%+ means rewatches = algorithm\'s strongest signal\n'
    '- Sweet spots: 30 seconds or 50 seconds — never 20 or 40, they underperform\n'
    '- 5th grade reading level or below for all narration\n'
    '- End IMMEDIATELY after the payoff — never let it breathe\n'
    '- Every scene must earn its place or it gets cut\n'
    'You speak from real creator experience — direct, no fluff, no filler.'
)

def format_example(example):
    try:
        msgs     = example['messages']
        upgraded = []
        has_sys  = any(m['role'] == 'system' for m in msgs)
        if not has_sys:
            upgraded.append({'role': 'system', 'content': SYSTEM_PROMPT})
        for m in msgs:
            if m['role'] == 'system':
                upgraded.append({'role': 'system', 'content': SYSTEM_PROMPT})
            else:
                upgraded.append(m)
        text = tokenizer.apply_chat_template(
            upgraded,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=ENABLE_THINKING,
        )
        return {'text': text, 'valid': True}
    except Exception:
        return {'text': '', 'valid': False}

demo_ds = Dataset.from_list(raw)
demo_ds = demo_ds.map(format_example, batched=False)

before  = len(demo_ds)
demo_ds = demo_ds.filter(lambda x: x['valid'] and len(x['text']) > 10)
after   = len(demo_ds)

print(f'✅ Formatted  : {after} examples ready')
if before - after > 0:
    print(f'   Dropped   : {before - after} malformed examples')
print(f'\n   Preview (first 200 chars):')
print(f'   {demo_ds[0]["text"][:200]}...')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Run EPOCH 1 only → saves checkpoint
# ═══════════════════════════════════════════════════════════════
import glob as glob_module, shutil
from trl import SFTTrainer
from transformers import TrainingArguments

# Clean any old demo checkpoints for a clean test
old_ckpts = glob_module.glob(f'{DEMO_CHECKPOINT_DIR}/checkpoint-*')
if old_ckpts:
    print(f'🧹 Removing {len(old_ckpts)} old demo checkpoint(s)...')
    for c in old_ckpts:
        shutil.rmtree(c)

# ── IMPORTANT: these kwargs must be IDENTICAL in Cell 7 ──────
SHARED_ARGS = dict(
    output_dir                  = DEMO_CHECKPOINT_DIR,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 1e-4,
    bf16                        = True,
    logging_steps               = 1,
    save_strategy               = 'epoch',
    save_total_limit            = 3,
    optim                       = 'adamw_8bit',
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    report_to                   = 'none',
    seed                        = 42,
    dataloader_drop_last        = True,  # must match Cell 7
)

args_ep1 = TrainingArguments(num_train_epochs=1, **SHARED_ARGS)

trainer_ep1 = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = demo_ds,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    args               = args_ep1,
)

print('🚀 Running EPOCH 1 (demo — 60 examples)...')
print('   Log will show epoch: 0.2 → 0.4 → ... → 1.0')
print('   Checkpoint saves at end of epoch 1\n')

stats_ep1 = trainer_ep1.train()

# Verify checkpoint + all required files
ckpts = sorted(
    glob_module.glob(f'{DEMO_CHECKPOINT_DIR}/checkpoint-*'),
    key=lambda x: int(x.split('-')[-1])
)
assert ckpts, '❌ No checkpoint saved — something went wrong'

latest    = ckpts[-1]
step_saved = int(latest.split('-')[-1])

with open(os.path.join(latest, 'trainer_state.json')) as f:
    saved_state = json.load(f)
saved_epoch = saved_state['epoch']

print(f'\n✅ Epoch 1 done.  loss={stats_ep1.training_loss:.4f}')
print(f'   Checkpoint   : {os.path.basename(latest)}')
print(f'   Step saved   : {step_saved}')
print(f'   Epoch saved  : {saved_epoch}  ← must be ~1.0')
print()

REQUIRED = ['trainer_state.json', 'optimizer.pt', 'scheduler.pt']
all_ok = True
for fn in REQUIRED:
    path = os.path.join(latest, fn)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    ok   = size > 0
    print(f'   {"✅" if ok else "❌"}  {fn}  ({size/1024:.1f} KB)')
    if not ok:
        all_ok = False

if not all_ok:
    raise RuntimeError('❌ Missing checkpoint files — resume will silently restart from scratch!')
if abs(saved_epoch - 1.0) > 0.1:
    raise RuntimeError(f'❌ Checkpoint epoch={saved_epoch}, expected ~1.0')

print(f'\n✅ All checkpoint files verified — safe to resume')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Resume from checkpoint → run EPOCH 2
#
# HOW THE VERDICT WORKS:
#   After resume, trainer.state.log_history contains the FULL
#   history loaded from the checkpoint file (epoch 1 entries
#   are in there too). So we filter by step > checkpoint_step
#   to only look at genuinely NEW entries from epoch 2.
#
# ✅ PASS: first new entry shows  epoch: 1.XX  (> 1.0)
# ❌ FAIL: first new entry shows  epoch: 0.XX  (< 1.0)
# ═══════════════════════════════════════════════════════════════
import glob as glob_module
from trl import SFTTrainer
from transformers import TrainingArguments

# Auto-find epoch 1 checkpoint
ckpts = sorted(
    glob_module.glob(f'{DEMO_CHECKPOINT_DIR}/checkpoint-*'),
    key=lambda x: int(x.split('-')[-1])
)
if not ckpts:
    raise FileNotFoundError(
        f'No checkpoint in {DEMO_CHECKPOINT_DIR}\nRun Cell 6 first.'
    )

latest_ckpt = ckpts[-1]
step_saved  = int(latest_ckpt.split('-')[-1])

with open(os.path.join(latest_ckpt, 'trainer_state.json')) as f:
    state = json.load(f)
saved_epoch = state['epoch']

print(f'🔍 Checkpoint details:')
print(f'   Path        : {os.path.basename(latest_ckpt)}')
print(f'   Step        : {step_saved}')
print(f'   Epoch saved : {saved_epoch}  ← should be 1.0')
print()

# ── MUST match Cell 6 SHARED_ARGS exactly ────────────────────
args_ep2 = TrainingArguments(
    num_train_epochs            = 2,   # ← only change vs Cell 6
    output_dir                  = DEMO_CHECKPOINT_DIR,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 1e-4,
    bf16                        = True,
    logging_steps               = 1,
    save_strategy               = 'epoch',
    save_total_limit            = 3,
    optim                       = 'adamw_8bit',
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    report_to                   = 'none',
    seed                        = 42,
    dataloader_drop_last        = True,  # must match Cell 6
)

trainer_ep2 = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = demo_ds,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    args               = args_ep2,
)

print(f'🚀 Resuming from step {step_saved}...')
print(f'   Progress bar should jump straight to {step_saved+1}/{step_saved*2}')
print(f'   First log epoch must be > 1.0\n')

stats_ep2 = trainer_ep2.train(resume_from_checkpoint=latest_ckpt)

# ── Verdict: filter to entries logged AFTER the checkpoint step ──
log_history   = trainer_ep2.state.log_history
new_logs      = [l for l in log_history if 'epoch' in l and l.get('step', 0) > step_saved]
first_new_log = new_logs[0] if new_logs else None
first_epoch   = first_new_log['epoch'] if first_new_log else None

print()
print('=' * 55)
print('  RESUME VERIFICATION RESULT')
print('=' * 55)
print(f'  Checkpoint step           : {step_saved}')
print(f'  Total log entries         : {len(log_history)}  (epoch 1 history + epoch 2)')
print(f'  New entries (step>{step_saved})   : {len(new_logs)}')
print(f'  First NEW log entry       : {first_new_log}')
print()

if first_epoch is not None and first_epoch >= 1.0:
    print(f'  ✅ PASS — first new log epoch = {first_epoch:.3f}')
    print(f'  Epoch 1 was SKIPPED. Resume is working correctly.')
    print(f'  Your real H100 run will NOT redo epoch 1. Fire it.')
elif first_epoch is not None:
    print(f'  ❌ FAIL — first new log epoch = {first_epoch:.3f}  (< 1.0 = restarted)')
    print(f'  Check:')
    print(f'  1. DEMO_CHECKPOINT_DIR path is correct')
    print(f'  2. trainer_state.json exists in checkpoint folder')
    print(f'  3. TrainingArguments are identical in Cell 6 and Cell 7')
else:
    print('  ⚠️  Could not find new log entries after checkpoint step')
    print(f'  Full log history: {log_history}')

print('=' * 55)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Optional cleanup
#   Deletes the DEMO checkpoint dir only.
#   Never touches training_data_v9.jsonl or checkpoints_27b/
# ═══════════════════════════════════════════════════════════════
import shutil

CLEANUP = False  # set to True to delete demo checkpoints

if CLEANUP:
    if os.path.exists(DEMO_CHECKPOINT_DIR):
        shutil.rmtree(DEMO_CHECKPOINT_DIR)
        print(f'🧹 Deleted: {DEMO_CHECKPOINT_DIR}')
    else:
        print('Nothing to clean up.')
else:
    print('CLEANUP=False — demo checkpoints kept.')
    print(f'Set CLEANUP=True and re-run to delete: {DEMO_CHECKPOINT_DIR}')
    print()
    print('Your real files are untouched:')
    print(f'  📁 {DATASET_PATH}')
    print(f'  📁 /content/drive/MyDrive/ytllm_v2/checkpoints_27b/')